# 04 — Content Recommendation System

**Goal:** Build a `recommend_movie('Stranger Things')` function that returns similar titles.

**Technique:** Content-based filtering with cosine similarity.

**Features used:** description (TF-IDF) + genre + cast + director

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from IPython.display import display

from src.data_cleaner import get_clean_df
from src.recommender import build_recommender

df = get_clean_df()
print(f'Dataset: {len(df):,} titles')

Loaded 8,807 rows × 12 columns
Dropped 0 duplicate rows
Cleaning complete ✅
type                object
year_added           Int64
duration_numeric     int64
primary_genre       object
Dataset: 8,807 titles


## Build the Recommender

In [2]:
# This takes ~30-60 seconds — building TF-IDF + combining all feature matrices
rec = build_recommender(df)
print('Recommender ready!')

Recommender fitted on 8,807 titles.
Feature matrix shape: (8807, 14042)
Recommender ready!


## Test: Recommend similar titles

In [3]:
# Test 1: Stranger Things
results = rec.recommend('Stranger Things', n=8)
print('=== Recommendations for: Stranger Things ===')
display(results[['title','type','listed_in','similarity_score']].style.background_gradient(cmap='Reds', subset=['similarity_score']))

=== Recommendations for: Stranger Things ===


,title,type,listed_in,similarity_score
0,Nightflyers,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",0.560200
1,Helix,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",0.540300
2,Chilling Adventures of Sabrina,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",0.528900
3,Manifest,TV Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",0.497100
4,Beyond Stranger Things,TV Show,"Stand-Up Comedy & Talk Shows, TV Mysteries, TV Sci-Fi & Fantasy",0.484700
5,Anjaan: Special Crimes Unit,TV Show,"International TV Shows, TV Horror, TV Mysteries",0.460900
6,Warrior Nun,TV Show,"TV Action & Adventure, TV Mysteries, TV Sci-Fi & Fantasy",0.451500
7,Spectros,TV Show,"International TV Shows, TV Horror, TV Mysteries",0.445900


In [4]:
# Test 2: The Dark Knight
results2 = rec.recommend('The Dark Knight', n=8)
print('=== Recommendations for: The Dark Knight ===')
display(results2[['title','type','listed_in','similarity_score']])

=== Recommendations for: The Dark Knight ===


,title,type,listed_in,similarity_score
0,Lunatics,TV Show,"International TV Shows, TV Comedies",0.5653
1,Yeh Meri Family,TV Show,"International TV Shows, TV Comedies",0.5207
2,Borges,TV Show,"International TV Shows, TV Comedies",0.5201
3,Masameer Classics,TV Show,"International TV Shows, TV Comedies",0.5194
4,Mahi Way,TV Show,"International TV Shows, TV Comedies",0.5124
5,Engineering Girls,TV Show,"International TV Shows, TV Comedies",0.5117
6,Show Me the Money,TV Show,"International TV Shows, TV Comedies",0.5111
7,Bibik-Bibikku,TV Show,"International TV Shows, TV Comedies",0.5103


In [5]:
# Test 3: Same-type only (TV Show → TV Show)
results3 = rec.recommend('Narcos', n=6, same_type=True)
print('=== Recommendations for: Narcos (TV Shows only) ===')
display(results3[['title','type','listed_in','similarity_score']])

=== Recommendations for: Narcos (TV Shows only) ===


,title,type,listed_in,similarity_score
0,Narcos: Mexico,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",0.5825
1,Marvel's Luke Cage,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",0.4961
2,Person of Interest,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",0.4924
3,Marvel's Jessica Jones,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",0.4894
4,Queen of the South,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",0.4834
5,Altered Carbon,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",0.4807


In [6]:
# Test 4: Text search
search_results = rec.search('space adventure robots science fiction', n=5)
print('=== Free-text search: "space adventure robots science fiction" ===')
display(search_results)

=== Free-text search: "space adventure robots science fiction" ===


,title,type,listed_in,description,score
0,Mars,TV Show,"Docuseries, Science & Nature TV, TV Dramas",Fact meets fiction in this docudrama chronicli...,0.2311
1,The Universe: Ancient Mysteries Solved,TV Show,"Docuseries, Science & Nature TV",From astronomical events to shapes and pattern...,0.2309
2,The Silence of the Marsh,Movie,"International Movies, Thrillers","While researching corruption for his new book,...",0.2199
3,Alien Worlds,TV Show,"British TV Shows, Docuseries, International TV...",Applying the laws of life on Earth to the rest...,0.2077
4,John & Jane,Movie,"Documentaries, International Movies",Truth and fiction blend in this quasi-document...,0.2077


## Evaluate: Manual spot-check

In [7]:
# Check title info
info = rec.get_title_info('Stranger Things')
print('Genre:', info['listed_in'])
print('Cast:', info['cast'][:100])
print('Description:', info['description'])

print()
print('Top recommendation:')
r = rec.recommend('Stranger Things', n=1).iloc[0]
print('Genre:', r['listed_in'])
print('Description:', r['description'])

Genre: TV Horror, TV Mysteries, TV Sci-Fi & Fantasy
Cast: Winona Ryder, David Harbour, Finn Wolfhard, Millie Bobby Brown, Gaten Matarazzo, Caleb McLaughlin, N
Description: When a young boy vanishes, a small town uncovers a mystery involving secret experiments, terrifying supernatural forces and one strange little girl.

Top recommendation:
Genre: TV Horror, TV Mysteries, TV Sci-Fi & Fantasy
Description: With humankind's future at stake, a group of scientists and a powerful telepath venture into the void aboard a spaceship full of secrets.


In [8]:
# Batch test — does the recommender make sense for different genres?
test_titles = ['Narcos', 'Bird Box', 'The Crown', 'Adam Sandler', 'Spirited Away']
for title in test_titles:
    try:
        r = rec.recommend(title, n=3)
        print(f"\n{title}:")
        for _, row in r.iterrows():
            print(f"  → {row['title']:30s} ({row['similarity_score']:.3f})  [{row['listed_in'][:50]}]")
    except ValueError as e:
        print(f"\n{title}: NOT FOUND - {e}")


Narcos:
  → Narcos: Mexico                 (0.583)  [Crime TV Shows, TV Action & Adventure, TV Dramas]
  → Marvel's Luke Cage             (0.496)  [Crime TV Shows, TV Action & Adventure, TV Dramas]
  → Person of Interest             (0.492)  [Crime TV Shows, TV Action & Adventure, TV Dramas]

Bird Box:
  → See You Yesterday              (0.485)  [Dramas, Sci-Fi & Fantasy, Thrillers]
  → Anon                           (0.455)  [Dramas, Sci-Fi & Fantasy, Thrillers]
  → In the Shadow of the Moon      (0.449)  [Dramas, Sci-Fi & Fantasy, Thrillers]

The Crown:
  → The A List                     (0.531)  [British TV Shows, International TV Shows, TV Drama]
  → Fate: The Winx Saga            (0.514)  [British TV Shows, International TV Shows, TV Drama]
  → Downton Abbey                  (0.513)  [British TV Shows, International TV Shows, TV Drama]

Adam Sandler:
  → Hari Kondabolu: Warn Your Relatives (0.472)  [Stand-Up Comedy]
  → The Stand-Up                   (0.471)  [Stand-Up Comedy]
  